In [15]:
import os
import re
import pickle
import shutil
import unicodedata
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("Notebook 11 initialized.")

Notebook 11 initialized.


In [16]:
# Cell 2 — locate project root

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

while not (
    (PROJECT_ROOT / "data" / "train").exists()
    and (PROJECT_ROOT / "output").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate amazon-ml-challenge project root."
        )

    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
OUTPUT_DIR = PROJECT_ROOT / "output"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

BLOCKING_PATH = OUTPUT_DIR / "blocking_artifacts_v1.pkl"
VAL_IDS_PATH = OUTPUT_DIR / "validation_s1_ids.txt"

print("Project:", PROJECT_ROOT)

Project: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge


In [17]:
with open(VAL_IDS_PATH, "r", encoding="utf-8") as f:
    validation_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

GT = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

GT_val = GT[
    GT["source1_entity_id"].isin(validation_s1_ids)
].copy()

true_targets_by_s1_id = {}

for row in GT_val.itertuples(index=False):
    if row.matched_entity_ids:
        true_targets_by_s1_id[row.source1_entity_id] = {
            x.strip()
            for x in row.matched_entity_ids.split(",")
            if x.strip()
        }
    else:
        true_targets_by_s1_id[row.source1_entity_id] = set()

print("Validation S1:", len(validation_s1_ids))
print("Validation GT rows:", len(GT_val))
print(
    "Validation true relationships:",
    sum(len(x) for x in true_targets_by_s1_id.values())
)



Validation S1: 220682
Validation GT rows: 220682
Validation true relationships: 764045


In [18]:
validation_s1_entity_ids = []

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):
        if row.entity_id in validation_s1_ids:
            validation_s1_entity_ids.append(row.entity_id)

assert len(validation_s1_entity_ids) == len(validation_s1_ids)
assert len(set(validation_s1_entity_ids)) == len(validation_s1_entity_ids)

val_s1_id_to_idx = {
    entity_id: idx
    for idx, entity_id in enumerate(validation_s1_entity_ids)
}

val_s1_idx_to_id = {
    idx: entity_id
    for entity_id, idx in val_s1_id_to_idx.items()
}

print("Validation S1 index:", len(val_s1_id_to_idx))
print("First 5:", validation_s1_entity_ids[:5])

Validation S1 index: 220682
First 5: ['S1-785847572', 'S1-796227187', 'S1-982210261', 'S1-791867209', 'S1-72918323']


In [19]:
with open(BLOCKING_PATH, "rb") as f:
    blocking = pickle.load(f)

token_to_id = blocking["token_to_id"]

name_exact_map = blocking["name_exact_map"]

name_token_to_s1 = blocking["name_token_to_s1"]
address_token_to_s1 = blocking["address_token_to_s1"]

name_pair_to_s1 = blocking["name_pair_to_s1"]
address_pair_to_s1 = blocking["address_pair_to_s1"]

active_name_pairs = blocking["active_name_pairs"]
active_address_pairs = blocking["active_address_pairs"]

target_name_pair_frequency_2 = blocking[
    "target_name_pair_frequency_2"
]

target_address_pair_frequency_2 = blocking[
    "target_address_pair_frequency_2"
]

NAME_TOKEN_THRESHOLD = blocking[
    "NAME_TOKEN_FREQUENCY_THRESHOLD"
]

ADDRESS_TOKEN_THRESHOLD = blocking[
    "ADDRESS_TOKEN_FREQUENCY_THRESHOLD"
]

PAIR_S1_THRESHOLD = blocking[
    "PAIR_S1_THRESHOLD"
]

TARGET_PAIR_THRESHOLD = blocking[
    "TARGET_PAIR_THRESHOLD"
]
name_token_frequency = blocking[
    "name_token_frequency"
]

address_token_frequency = blocking[
    "address_token_frequency"
]

print("Blocker loaded.")
print(
    "Thresholds:",
    NAME_TOKEN_THRESHOLD,
    ADDRESS_TOKEN_THRESHOLD,
    PAIR_S1_THRESHOLD,
    TARGET_PAIR_THRESHOLD
)

Blocker loaded.
Thresholds: 10 10 2 500


In [20]:
MODEL_DIR = OUTPUT_DIR / "models_v1"

with open(
    MODEL_DIR / "logistic_model.pkl",
    "rb"
) as f:
    logistic_model = pickle.load(f)

with open(
    MODEL_DIR / "logistic_scaler.pkl",
    "rb"
) as f:
    logistic_scaler = pickle.load(f)

with open(
    MODEL_DIR / "xgboost_model.pkl",
    "rb"
) as f:
    xgb_model = pickle.load(f)

with open(
    MODEL_DIR / "lightgbm_model.pkl",
    "rb"
) as f:
    lgb_model = pickle.load(f)

with open(
    MODEL_DIR / "model_feature_names.pkl",
    "rb"
) as f:
    model_feature_names = pickle.load(f)

print("Models loaded.")
print("Feature count:", len(model_feature_names))
print(model_feature_names)

Models loaded.
Feature count: 16
['name_exact', 'name_ratio', 'name_token_ratio', 'name_token_jaccard', 'name_length_ratio', 'name_digit_overlap', 'address_exact', 'address_ratio', 'address_token_ratio', 'address_token_jaccard', 'address_length_ratio', 'address_digit_overlap', 'country_same', 'name_present_both', 'address_present_both', 'source_is_s3']


In [21]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize(
        "NFKC",
        value
    )

    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def get_tokens(value):
    normalized = normalize_text(value)

    if not normalized:
        return []

    return normalized.split()


def get_digit_tokens(value):
    if not value:
        return set()

    return set(
        re.findall(
            r"\d+",
            value
        )
    )


def length_ratio(a, b):
    if not a or not b:
        return 0.0

    return (
        min(len(a), len(b))
        /
        max(len(a), len(b))
    )


def token_jaccard(a, b):
    if not a and not b:
        return 0.0

    union = a | b

    if not union:
        return 0.0

    return len(a & b) / len(union)

In [22]:
N_VAL_S1 = len(validation_s1_entity_ids)

val_s1_name = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_address = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_country = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_name_tokens = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_address_tokens = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_name_digits = np.empty(
    N_VAL_S1,
    dtype=object
)

val_s1_address_digits = np.empty(
    N_VAL_S1,
    dtype=object
)

filled = np.zeros(
    N_VAL_S1,
    dtype=bool
)

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):

        idx = val_s1_id_to_idx.get(
            row.entity_id
        )

        if idx is None:
            continue

        name = normalize_text(
            row.business_name
        )

        address = normalize_text(
            row.business_address
        )

        country = row.country.casefold().strip()

        val_s1_name[idx] = name
        val_s1_address[idx] = address
        val_s1_country[idx] = country

        val_s1_name_tokens[idx] = set(
            get_tokens(name)
        )

        val_s1_address_tokens[idx] = set(
            get_tokens(address)
        )

        val_s1_name_digits[idx] = (
            get_digit_tokens(name)
        )

        val_s1_address_digits[idx] = (
            get_digit_tokens(address)
        )

        filled[idx] = True

assert filled.all()

print(
    "Validation S1 records loaded:",
    int(filled.sum())
)

Validation S1 records loaded: 220682


In [27]:
def generate_validation_candidates(
    target_name,
    target_address
):
    candidate_s1 = set()

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    normalized_name = normalize_text(
        target_name
    )

    normalized_address = normalize_text(
        target_address
    )

    # --------------------------------------------------------
    # Token IDs
    # --------------------------------------------------------

    name_tokens = set(
        get_tokens(normalized_name)
    )

    address_tokens = set(
        get_tokens(normalized_address)
    )

    name_token_ids = sorted(
        token_to_id[token]
        for token in name_tokens
        if token in token_to_id
    )

    address_token_ids = sorted(
        token_to_id[token]
        for token in address_tokens
        if token in token_to_id
    )

    # --------------------------------------------------------
    # 1. Exact normalized name
    # --------------------------------------------------------

    if normalized_name:
        candidate_s1.update(
            name_exact_map.get(
                normalized_name,
                []
            )
        )

    # --------------------------------------------------------
    # 2. Name-token blocking
    # IMPORTANT: only use tokens with frequency <= threshold
    # --------------------------------------------------------

    for token in name_tokens:

        freq = name_token_frequency.get(
            token,
            0
        )

        if 0 < freq <= NAME_TOKEN_THRESHOLD:
            candidate_s1.update(
                name_token_to_s1.get(
                    token,
                    []
                )
            )

    # --------------------------------------------------------
    # 3. Address-token blocking
    # IMPORTANT: only use tokens with frequency <= threshold
    # --------------------------------------------------------

    for token in address_tokens:

        freq = address_token_frequency.get(
            token,
            0
        )

        if 0 < freq <= ADDRESS_TOKEN_THRESHOLD:
            candidate_s1.update(
                address_token_to_s1.get(
                    token,
                    []
                )
            )

    # --------------------------------------------------------
    # 4. S1-side name pair blocking
    # --------------------------------------------------------

    if len(name_token_ids) >= 2:

        for pair in combinations(
            name_token_ids,
            2
        ):

            if pair in active_name_pairs:
                candidate_s1.update(
                    name_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    # --------------------------------------------------------
    # 5. S1-side address pair blocking
    # --------------------------------------------------------

    if len(address_token_ids) >= 2:

        for pair in combinations(
            address_token_ids,
            2
        ):

            if pair in active_address_pairs:
                candidate_s1.update(
                    address_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    # --------------------------------------------------------
    # 6. Target-side name pair <= TARGET_PAIR_THRESHOLD
    # --------------------------------------------------------

    if len(name_token_ids) >= 2:

        for pair in combinations(
            name_token_ids,
            2
        ):

            freq = target_name_pair_frequency_2.get(
                pair
            )

            if (
                freq is not None
                and 0 < freq <= TARGET_PAIR_THRESHOLD
            ):
                candidate_s1.update(
                    name_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    # --------------------------------------------------------
    # 7. Target-side address pair <= TARGET_PAIR_THRESHOLD
    # --------------------------------------------------------

    if len(address_token_ids) >= 2:

        for pair in combinations(
            address_token_ids,
            2
        ):

            freq = target_address_pair_frequency_2.get(
                pair
            )

            if (
                freq is not None
                and 0 < freq <= TARGET_PAIR_THRESHOLD
            ):
                candidate_s1.update(
                    address_pair_to_s1.get(
                        pair,
                        []
                    )
                )

    return sorted(candidate_s1)

In [28]:
def build_pair_features(
    candidate_indices,
    target_name,
    target_address,
    target_country,
    source_number
):
    target_name = normalize_text(
        target_name
    )

    target_address = normalize_text(
        target_address
    )

    target_country = (
        target_country.casefold().strip()
    )

    target_name_tokens = set(
        get_tokens(target_name)
    )

    target_address_tokens = set(
        get_tokens(target_address)
    )

    target_name_digits = (
        get_digit_tokens(target_name)
    )

    target_address_digits = (
        get_digit_tokens(target_address)
    )

    n = len(candidate_indices)

    X = np.empty(
        (n, 16),
        dtype=np.float32
    )

    for j, s1_idx_value in enumerate(
        candidate_indices
    ):

        s1_n = val_s1_name[s1_idx_value]
        s1_a = val_s1_address[s1_idx_value]
        s1_c = val_s1_country[s1_idx_value]

        s1_nt = val_s1_name_tokens[
            s1_idx_value
        ]

        s1_at = val_s1_address_tokens[
            s1_idx_value
        ]

        s1_nd = val_s1_name_digits[
            s1_idx_value
        ]

        s1_ad = val_s1_address_digits[
            s1_idx_value
        ]

        name_exact = int(
            bool(s1_n)
            and s1_n == target_name
        )

        name_ratio = (
            fuzz.ratio(
                s1_n,
                target_name
            ) / 100.0
            if s1_n and target_name
            else 0.0
        )

        name_token_ratio = (
            fuzz.token_set_ratio(
                s1_n,
                target_name
            ) / 100.0
            if s1_n and target_name
            else 0.0
        )

        name_jaccard = token_jaccard(
            s1_nt,
            target_name_tokens
        )

        name_len_ratio = length_ratio(
            s1_n,
            target_name
        )

        name_digit_overlap = len(
            s1_nd
            &
            target_name_digits
        )

        address_exact = int(
            bool(s1_a)
            and s1_a == target_address
        )

        address_ratio = (
            fuzz.ratio(
                s1_a,
                target_address
            ) / 100.0
            if s1_a and target_address
            else 0.0
        )

        address_token_ratio = (
            fuzz.token_set_ratio(
                s1_a,
                target_address
            ) / 100.0
            if s1_a and target_address
            else 0.0
        )

        address_jaccard = token_jaccard(
            s1_at,
            target_address_tokens
        )

        address_len_ratio = length_ratio(
            s1_a,
            target_address
        )

        address_digit_overlap = len(
            s1_ad
            &
            target_address_digits
        )

        country_same = int(
            bool(s1_c)
            and s1_c == target_country
        )

        name_present_both = int(
            bool(s1_n and target_name)
        )

        address_present_both = int(
            bool(s1_a and target_address)
        )

        X[j] = [
            name_exact,
            name_ratio,
            name_token_ratio,
            name_jaccard,
            name_len_ratio,
            name_digit_overlap,

            address_exact,
            address_ratio,
            address_token_ratio,
            address_jaccard,
            address_len_ratio,
            address_digit_overlap,

            country_same,
            name_present_both,
            address_present_both,

            1.0 if source_number == 3 else 0.0
        ]

    return X